# Permuting and sampling data to calculate the null hypothesis

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.tsv') and ('null-corpus' not in f)]
print(files)

['../data/server_ready/cha_corpus-callfriend.tsv', '../data/server_ready/cha_corpus-callhome.tsv', '../data/server_ready/callfriend-ko_corpus.tsv', '../data/server_ready/cha_corpus-yiddish.tsv', '../data/server_ready/d_cha_corpus-croatian.tsv', '../data/server_ready/finchat_corpus.tsv']


In [3]:
null_output_file = 'null-corpora/{}-null-corpus_permuted-within-convo.tsv'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

'../data/server_ready/null-corpora/{}-null-corpus_permuted-within-convo.tsv'

## Setting up the null-test data

### Load, permute, save

In [5]:
for f in tqdm(files):
    print(f)
    # df = pd.read_csv(f)
    df = pd.read_table(f, sep='\t')

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
    ]
    # print(df[['x_utterance', 'y_utterance']].isna().sum())

    df['x_turn_id'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)

    df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1


    print('finding examples')
    # a lot of work just to find useful examples ...
    good_examples = df[['corpus', 'x_turn_id', 'self', 'sample_delta']].groupby(by=['corpus', 'x_turn_id', 'self', 'sample_delta']).agg('max').reset_index()
    good_examples = pd.merge(
        left=good_examples.loc[good_examples['self']], left_on='x_turn_id',
        right=good_examples[['x_turn_id', 'self', 'sample_delta']].loc[~good_examples['self']], right_on='x_turn_id',
        how='left'
    )
    good_examples = good_examples.loc[
        ((good_examples['sample_delta_x'] >= 5)
        & (good_examples['sample_delta_x'] <= 70))
        & ((good_examples['sample_delta_y'] >= 5)
        & (good_examples['sample_delta_y'] <= 70))
    ]

    df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    print('permuting sentences')
    for cid in tqdm(df['conversation_id'].unique()):
        sub = df.loc[df['conversation_id'].isin([cid])]
        df['y_utterance'].loc[sub.index] = sub['y_utterance'].sample(frac=1.0)

    print('saving data')
    for c in tqdm(df['corpus'].unique()):

        sub = df['x_turn_id'].loc[df['corpus'].isin([c])].unique()

        k = int(np.ceil(len(sub)*.1))

        if k > 0:
            sub = np.random.choice(sub, size=(k,), replace=False)
            df_ = df.loc[df['x_turn_id'].isin(sub)]

            print('\n', c, k, df_.shape, '\n', df_[['x_utterance', 'y_utterance']].isna().sum())

            null_output_file_ = str(null_output_file).format(c)
            # save subset to null output file.
            # if os.path.exists(null_output_file):
            #     df_.to_csv(null_output_file, index=False, header=False, encoding='utf-8', mode='a', sep='\t')
            # else:
            df_.to_csv(null_output_file_, index=False, encoding='utf-8', sep='\t')
    print('===][===')

  0%|          | 0/6 [00:00<?, ?it/s]

../data/server_ready/cha_corpus-callfriend.tsv
finding examples
permuting sentences


  0%|          | 0/205 [00:00<?, ?it/s]

saving data


  0%|          | 0/4 [00:00<?, ?it/s]

...


 croation-cro 5675 (325933, 21) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===
../data/server_ready/finchat_corpus.tsv
finding examples
permuting sentences


  0%|          | 0/79 [00:00<?, ?it/s]

saving data


  0%|          | 0/1 [00:00<?, ?it/s]


 finchat-fin 263 (8868, 13) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===


### test that file works

In [6]:
corpora_are_in = os.path.join(data_loc, 'null-corpora')
corpora_are_in

'../data/server_ready/null-corpora'

In [7]:
corpora = [f for f in os.listdir(corpora_are_in) if (not f.startswith('._')) and f.endswith('.tsv')]
corpora

['callfriend-eng-n-null-corpus_permuted-within-convo.tsv',
 'callfriend-eng-s-null-corpus_permuted-within-convo.tsv',
 'callfriend-fra-q-null-corpus_permuted-within-convo.tsv',
 'callfriend-spa-s-null-corpus_permuted-within-convo.tsv',
 'callhome-deu-null-corpus_permuted-within-convo.tsv',
 'callhome-eng-null-corpus_permuted-within-convo.tsv',
 'callhome-jpn-null-corpus_permuted-within-convo.tsv',
 'callhome-spa-null-corpus_permuted-within-convo.tsv',
 'callhome-zho-null-corpus_permuted-within-convo.tsv',
 'callfriend-ko-null-corpus_permuted-within-convo.tsv',
 'yiddish-yid-men-null-corpus_permuted-within-convo.tsv',
 'croation-cro-null-corpus_permuted-within-convo.tsv',
 'finchat-fin-null-corpus_permuted-within-convo.tsv']

In [8]:
df = []

In [9]:
for f in corpora:
    df += [pd.read_table(os.path.join(corpora_are_in,f), sep='\t')]
    # print(f, df[-1].shape)
    # print(df[-1][['x_utterance', 'y_utterance']].isna().sum())
    # print('===][===\n')
df = pd.concat(df, ignore_index=True)

In [10]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [11]:
null_output_file = os.path.join(data_loc,'null-corpus_permuted-within-convo.tsv')

In [12]:
df.to_csv(null_output_file, index=False, encoding='utf-8', sep='\t')

#### Test that stitched doc is okay

In [13]:
null_output_file = os.path.join(data_loc,'null-corpus_permuted-within-convo.tsv')

In [14]:
# df = pd.read_csv(null_output_file)
df = pd.read_table(null_output_file, sep='\t')
df.shape

(2553608, 27)

In [15]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [16]:
df.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,overlapping_text,corpus,conversation_id,com,...,mor,gra,exp,time,err,sit,add,act,wor,TIME
0,28.0,callfriend-eng-n-4175-9,callfriend-eng-n-callfriend-eng-n-4175-M1,hhh well the xxx was,"[6496, 7616]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,28.0,callfriend-eng-n-4175-9,callfriend-eng-n-callfriend-eng-n-4175-M1,hhh well the xxx was,"[6496, 7616]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,28.0,callfriend-eng-n-4175-9,callfriend-eng-n-callfriend-eng-n-4175-M1,hhh well the xxx was,"[6496, 7616]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28.0,callfriend-eng-n-4175-9,callfriend-eng-n-callfriend-eng-n-4175-M1,hhh well the xxx was,"[6496, 7616]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,28.0,callfriend-eng-n-4175-9,callfriend-eng-n-callfriend-eng-n-4175-M1,hhh well the xxx was,"[6496, 7616]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
